In [ ]:
# ============================================================
# Two-LLM simulation (Roleplayer + Agent) for weighted relaxation
# REIMPLEMENTED to use: google/gemma-2-9b-it
# - Uses Gemma 2 IT chat template via tokenizer.apply_chat_template()
# - Gemma chat roles are "user" and "model" (no separate "system" role in HF template),
#   so we inline the "system prompt" into the user content.
# ============================================================

import os, json, re
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from setproctitle import setproctitle
setproctitle("python")

# ----------------------------
# Environment / GPUs
# ----------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "4"   # only use physical GPUs 2 and 3
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ----------------------------
# Paths / config
# ----------------------------
BENCH_PATH_CANDS = ["./44fix.json", "/mnt/data/queries_10.json"]
DATA_PATH_CANDS  = ["./data.csv", "/mnt/data/data.csv"]

AGENT_OUT_PATH = "./gamma/Gamma_weighted44.json"

MODEL_NAME = "google/gemma-2-9b-it"
SEED = 11
rng = np.random.default_rng(SEED)

EXCLUDE = {"Engine HP", "Engine Cylinders", "Market Category", "Popularity", "Number of Doors"}
BASE_COLS = ["Make", "Transmission Type", "Driven_Wheels", "Vehicle Style"]

NUMERIC_COLS = {"Year", "city mpg", "highway MPG", "MSRP"}
CATEGORICAL_COLS = {"Engine Fuel Type", "Vehicle Size", "Model"}
ALLOWED_EXTRA_COLS = list(NUMERIC_COLS | CATEGORICAL_COLS)
ALLOWED_OPS = {"==", ">=", "<="}

MAX_DIALOG_TURNS_PER_SLOT = 1          # one Q + one A per slot (keeps sim simple/cheap)
MAX_NEW_TOKENS_Q = 120
MAX_NEW_TOKENS_A = 200
MAX_NEW_TOKENS_PARSE = 240

# ----------------------------
# Load benchmark + data
# ----------------------------
bench_path = next((p for p in BENCH_PATH_CANDS if os.path.exists(p)), None)
if bench_path is None:
    raise FileNotFoundError("Could not find queries_10.json in ./ or /mnt/data/")

data_path = next((p for p in DATA_PATH_CANDS if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError("Could not find data.csv in ./ or /mnt/data/")

with open(bench_path, "r", encoding="utf-8") as f:
    bench = json.load(f)

df0 = pd.read_csv(data_path)
df = df0[[c for c in df0.columns if c not in EXCLUDE]].copy()
if "Engine Fuel Type" in df.columns:
    df["Engine Fuel Type"] = df["Engine Fuel Type"].fillna("Unknown")

# ----------------------------
# Canonicalization helpers
# ----------------------------
def _build_vocab_maps(df: pd.DataFrame, cols: List[str]):
    vocab_list = {}
    vocab_lower_to_orig = {}
    for c in cols:
        if c not in df.columns:
            continue
        vals = df[c].dropna().astype(str).unique().tolist()
        vocab_list[c] = vals
        m = {}
        for v in vals:
            vl = v.strip().lower()
            if vl and vl not in m:
                m[vl] = v
        vocab_lower_to_orig[c] = m
    return vocab_list, vocab_lower_to_orig

VOCAB_LIST, VOCAB_LOWER_TO_ORIG = _build_vocab_maps(df, list(CATEGORICAL_COLS))
MAKE_SET_LOWER = set(df["Make"].dropna().astype(str).str.strip().str.lower().unique().tolist()) if "Make" in df.columns else set()

def canonicalize_categorical(col: str, raw: str) -> str:
    """
    Map a possibly noisy categorical string to the dataset's exact vocabulary.
    Handles cases like "Dodge Stratus" -> "Stratus" (Model), casing differences,
    and substring/contains matches.
    """
    s = (raw or "").strip()
    if s == "" or col not in VOCAB_LOWER_TO_ORIG:
        return s

    low = s.lower()
    m = VOCAB_LOWER_TO_ORIG[col]
    if low in m:
        return m[low]

    # For Model: drop leading Make token if present
    s2 = s
    low2 = low
    if col == "Model":
        toks = re.split(r"\s+", s)
        if toks:
            t0 = toks[0].strip().lower()
            if t0 in MAKE_SET_LOWER and len(toks) >= 2:
                s2 = " ".join(toks[1:]).strip()
                low2 = s2.lower()
                if low2 in m:
                    return m[low2]

    def _clean(x: str) -> str:
        x = x.lower()
        x = re.sub(r"[^a-z0-9\s\-\/]", " ", x)
        x = re.sub(r"\b(the|a|an|model|trim|edition)\b", " ", x)
        x = re.sub(r"\s+", " ", x).strip()
        return x

    c_low = _clean(low2)

    best = None
    best_len = -1
    for v in VOCAB_LIST.get(col, []):
        vl = _clean(str(v))
        if not vl:
            continue
        if vl in c_low or c_low in vl:
            if len(vl) > best_len:
                best = v
                best_len = len(vl)
    if best is not None:
        return str(best)

    return s

# ----------------------------
# Gemma 2 load + robust JSON extraction
# ----------------------------
def load_gemma2(model_name: str = MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Ensure pad token
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Prefer bfloat16 (native weights), fallback to float16 if needed
    dtype_try = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=dtype_try,
        )
    except Exception:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        )

    model.eval()
    return tokenizer, model

tok, mdl = load_gemma2(MODEL_NAME)

def _extract_json_robust(text: str) -> dict:
    t = text.strip()
    # fenced block preference
    if "```" in t:
        parts = t.split("```")
        for i in range(len(parts)-1):
            fence = parts[i].strip().lower()
            payload = parts[i+1].strip()
            if "json" in fence or fence == "":
                lines = payload.splitlines()
                if lines and lines[0].strip().lower() == "json":
                    payload = "\n".join(lines[1:]).strip()
                try:
                    return json.loads(payload)
                except Exception:
                    t = payload
                    break
    start = t.find("{")
    if start == -1:
        raise ValueError("No '{' found in model output.")
    depth = 0
    for i in range(start, len(t)):
        ch = t[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return json.loads(t[start:i+1])
    end = t.rfind("}")
    if end != -1 and end > start:
        return json.loads(t[start:end+1])
    raise ValueError("Failed to parse JSON object from model output.")

def _gemma_apply_chat(user_content: str) -> str:
    """
    Gemma-2-*-it HF chat template uses roles: user/model.
    HF template often doesn't support a distinct "system" role, so we inline system text.
    """
    chat = [{"role": "user", "content": user_content}]
    try:
        return tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    except Exception:
        # Fallback manual format from model card:
        # <bos><start_of_turn>user\n...<end_of_turn>\n<start_of_turn>model\n
        bos = tok.bos_token or "<bos>"
        return f"{bos}<start_of_turn>user\n{user_content}<end_of_turn>\n<start_of_turn>model\n"

@torch.inference_mode()
def gemma_gen_text(system_prompt: str, user_prompt: str, max_new_tokens: int = 200) -> str:
    # Inline "system" instructions into user content (Gemma HF template typically uses user/model only).
    combined = (system_prompt or "").strip()
    if combined:
        combined = combined + "\n\n" + (user_prompt or "").strip()
    else:
        combined = (user_prompt or "").strip()

    prompt = _gemma_apply_chat(combined)

    # Tokenize WITHOUT adding special tokens again (prompt already includes them)
    inputs = tok(prompt, return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(mdl.device) for k, v in inputs.items()}
    if "attention_mask" not in inputs or inputs["attention_mask"] is None:
        inputs["attention_mask"] = (inputs["input_ids"] != tok.pad_token_id).long()

    prompt_len = inputs["input_ids"].shape[-1]
    out = mdl.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
    )
    gen_ids = out[0][prompt_len:]
    return tok.decode(gen_ids, skip_special_tokens=True).strip()

@torch.inference_mode()
def gemma_gen_json(system_prompt: str, user_prompt: str, max_new_tokens: int = 260, retries: int = 3) -> dict:
    strict_suffixes = [
        "",
        "\n\nIMPORTANT: Output MUST start with '{' and end with '}' and contain NOTHING else.",
        "\n\nFINAL WARNING: ONLY output JSON. No markdown. No code fences. No commentary."
    ]
    last_text, last_err = None, None
    for attempt in range(retries):
        sys = (system_prompt or "") + strict_suffixes[min(attempt, len(strict_suffixes)-1)]
        txt = gemma_gen_text(sys, user_prompt, max_new_tokens=max_new_tokens)
        last_text = txt
        try:
            return _extract_json_robust(txt)
        except Exception as e:
            last_err = e
            continue
    raise ValueError(f"Could not parse JSON after {retries} tries. Last error: {last_err}\n\nLast model output:\n{last_text}")

# ----------------------------
# Sanitizers
# ----------------------------
def coerce_int(v) -> Optional[int]:
    if v is None:
        return None
    if isinstance(v, (int, np.integer)):
        return int(v)
    if isinstance(v, float) and np.isfinite(v):
        return int(round(v))
    s = str(v).strip()
    m = re.search(r"-?\d+", s)
    if not m:
        return None
    try:
        return int(m.group(0))
    except Exception:
        return None

def sanitize_constraint(p: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    if not isinstance(p, dict):
        return None
    col, op, val = p.get("column"), p.get("op"), p.get("value")
    if col not in ALLOWED_EXTRA_COLS or op not in ALLOWED_OPS:
        return None
    if col in NUMERIC_COLS:
        if op in (">=", "<=", "=="):
            iv = coerce_int(val)
            if iv is None:
                return None
            return {"column": col, "op": op, "value": iv}
        return None
    if col in CATEGORICAL_COLS:
        if op != "==":
            return None
        sval = str(val).strip()
        if sval == "":
            return None
        sval = canonicalize_categorical(col, sval)
        if sval == "":
            return None
        return {"column": col, "op": "==", "value": sval}
    return None

def dedup_constraints(extras: List[Dict[str,Any]]) -> List[Dict[str,Any]]:
    seen = set()
    out = []
    for p in extras:
        k = (p["column"], p["op"], str(p["value"]))
        if k in seen:
            continue
        seen.add(k)
        out.append(p)
    return out

# ----------------------------
# "Solver" filtering
# ----------------------------
def apply_constraints(df: pd.DataFrame, base: Dict[str,str], extras: List[Dict[str,Any]]) -> pd.DataFrame:
    m = np.ones(len(df), dtype=bool)
    for c in BASE_COLS:
        m &= (df[c].values == base[c])
    for p in extras:
        col, op, val = p["column"], p["op"], p["value"]
        if col not in df.columns:
            return df.iloc[0:0].copy()
        if op == "==":
            m &= (df[col].astype(str).values == str(val))
        elif op == ">=":
            m &= (df[col].values >= int(val))
        elif op == "<=":
            m &= (df[col].values <= int(val))
        else:
            return df.iloc[0:0].copy()
    return df[m].copy()

# ----------------------------
# Weighting + recommendation
# ----------------------------
def normalize_series(x: np.ndarray) -> np.ndarray:
    xmin = float(np.min(x))
    xmax = float(np.max(x))
    if xmax - xmin < 1e-9:
        return np.zeros_like(x, dtype=float)
    return (x - xmin) / (xmax - xmin)

def recommend_after_relaxation(
    U: pd.DataFrame,
    additional: List[Dict[str,Any]],
    weights: Dict[Tuple[str,str,str], float],
    relaxed_key: Tuple[str,str,str]
) -> Optional[Dict[str,Any]]:
    keep = [p for p in additional if (p["column"], p["op"], str(p["value"])) != relaxed_key]
    m = np.ones(len(U), dtype=bool)
    for p in keep:
        col, op, val = p["column"], p["op"], p["value"]
        if op == "==":
            m &= (U[col].astype(str).values == str(val))
        elif op == ">=":
            m &= (U[col].values >= int(val))
        elif op == "<=":
            m &= (U[col].values <= int(val))
    cand = U[m].copy()
    if len(cand) == 0:
        return None

    scores = np.zeros(len(cand), dtype=float)
    norms = {}
    for col in ["Year","city mpg","highway MPG","MSRP"]:
        if col in U.columns:
            norms[col] = normalize_series(U[col].values)

    cand_pos = cand.index.values
    for p in additional:
        k = (p["column"], p["op"], str(p["value"]))
        w = float(weights.get(k, 0.25))
        col, op, val = p["column"], p["op"], p["value"]
        if op == "==":
            sat = (cand[col].astype(str).values == str(val)).astype(float)
            scores += w * sat
        elif op == ">=":
            if col in norms:
                scores += w * norms[col][cand_pos]
            else:
                scores += w * (cand[col].values >= int(val)).astype(float)
        elif op == "<=":
            if col in norms:
                scores += w * (1.0 - norms[col][cand_pos])
            else:
                scores += w * (cand[col].values <= int(val)).astype(float)

    msrp = cand["MSRP"].values.astype(float) if "MSRP" in cand.columns else np.zeros(len(cand))
    best_idx = np.lexsort((msrp, -scores))[0]
    row = cand.iloc[best_idx]

    def gi(k):
        return int(row[k]) if k in cand.columns and pd.notna(row[k]) else None

    return {
        "Make": row.get("Make", None),
        "Model": row.get("Model", None),
        "Year": gi("Year"),
        "Engine Fuel Type": row.get("Engine Fuel Type", None),
        "Transmission Type": row.get("Transmission Type", None),
        "Driven_Wheels": row.get("Driven_Wheels", None),
        "Vehicle Size": row.get("Vehicle Size", None),
        "Vehicle Style": row.get("Vehicle Style", None),
        "city mpg": gi("city mpg"),
        "highway MPG": gi("highway MPG"),
        "MSRP": gi("MSRP"),
    }

# ----------------------------
# Two-LLM simulation: Roleplayer answers, Agent asks + parses + weights
# ----------------------------
ROLEPLAYER_SYS = (
    "You are ROLEPLAYER_USER. You ONLY know the provided persona and the base query. "
    "Answer the assistant's question using ONLY information that is stated or clearly implied in the persona. "
    "IMPORTANT: When the assistant is clarifying a constraint, you should be helpful and specific: "
    "if the persona provides ANY relevant value (e.g., a specific model name, a year minimum, a budget cap), "
    "you MUST state that value explicitly in your answer. "
    "Do NOT say 'no preference' or omit the value if the persona mentions it. "
    "Only say you have no preference if the persona truly contains no information about that attribute. "
    "If the assistant's question is a bit vague, still provide the most relevant constraint value(s) from the persona. "
    "Do not invent details that are not in the persona."
)

AGENT_ASK_SYS = (
    "You are CLARIFYING_AGENT. Ask ONE short question to learn the user's requirement for the given slot/field. "
    "Do not propose relaxations. Do not ask multiple questions at once."
)

AGENT_EXTRACT_SYS = (
    "You are a strict JSON generator. Extract (1) the constraint value from the user's answer for the given slot, "
    "and (2) an importance weight in [0,1] where 1=very important (must-have) and 0=very flexible (easy to relax). "
    "Output ONLY JSON."
)

def agent_question(slot: str, base_query: str, history: List[Dict[str,str]]) -> str:
    hist_txt = "\n".join([f"{m['role'].upper()}: {m['content']}" for m in history[-6:]])
    user_prompt = f"""
Base query: {base_query}

Conversation so far:
{hist_txt}

Slot to clarify: {slot}

Ask ONE question to determine the user's requirement for {slot}. Keep it concise.
""".strip()
    return gemma_gen_text(AGENT_ASK_SYS, user_prompt, max_new_tokens=MAX_NEW_TOKENS_Q)

def roleplayer_answer(persona: str, base_query: str, question: str) -> str:
    user_prompt = f"""
Base query the user started with:
{base_query}

Persona (all you know):
{persona}

Assistant asked:
{question}

Answer as the user, using only persona information.
""".strip()
    return gemma_gen_text(ROLEPLAYER_SYS, user_prompt, max_new_tokens=MAX_NEW_TOKENS_A)

def extract_constraint_and_weight(slot: str, user_answer: str) -> Tuple[Optional[Dict[str,Any]], float]:
    user_prompt = f"""
Slot: {slot}
User answer: {user_answer}

Return JSON:
{{
  "constraint": null OR {{"column": "{slot}", "op": "=="|">="|"<="| "==", "value": string|number}},
  "weight": number
}}

Rules:
- If user has no preference / doesn't care, output constraint:null and weight 0.0.
- If slot is numeric (Year, city mpg, highway MPG, MSRP):
  - Use >= for minimum requirements, <= for maximum budget, == only if explicitly exact.
  - value MUST be an integer.
- If slot is categorical (Engine Fuel Type, Vehicle Size, Model):
  - Use op == and value as string.
- weight in [0,1]: must-have ~0.9-1.0; strongly prefer ~0.7-0.9; nice-to-have ~0.4-0.7; flexible/compromise ~0.0-0.3.
Output ONLY JSON.
""".strip()
    obj = gemma_gen_json(AGENT_EXTRACT_SYS, user_prompt, max_new_tokens=MAX_NEW_TOKENS_PARSE, retries=3)
    w = float(obj.get("weight", 0.5))
    w = max(0.0, min(1.0, w))
    c = obj.get("constraint", None)
    if c is None:
        return None, w
    c["column"] = slot  # force slot alignment
    sc = sanitize_constraint(c)
    if sc is None:
        return None, w
    return sc, w

def parse_base_from_sentence(df: pd.DataFrame, base_sentence: str) -> Dict[str,str]:
    s = base_sentence.lower()
    def match_vocab(col: str) -> str:
        vocab = sorted(df[col].dropna().astype(str).unique().tolist(), key=len, reverse=True)
        for v in vocab:
            if v.lower() in s:
                return v
        raise ValueError(f"Could not parse base field {col} from: {base_sentence}")
    return {
        "Make": match_vocab("Make"),
        "Vehicle Style": match_vocab("Vehicle Style"),
        "Transmission Type": match_vocab("Transmission Type"),
        "Driven_Wheels": match_vocab("Driven_Wheels"),
    }

# ----------------------------
# Run simulation for all benchmark queries
# ----------------------------
outputs = []

for i, b in enumerate(bench, 1):
    base_query = b["base_query_sentence"]
    persona = b.get("persona", "")

    slots = sorted({c["column"] for c in b.get("additional_constraints", [])})

    history = [{"role":"user", "content": base_query}]
    collected_constraints: List[Dict[str,Any]] = []
    weights: Dict[Tuple[str,str,str], float] = {}

    for slot in slots:
        q = agent_question(slot, base_query, history)
        history.append({"role":"assistant","content": q})

        a = roleplayer_answer(persona, base_query, q)
        history.append({"role":"user","content": a})

        c, w = extract_constraint_and_weight(slot, a)

        if c is not None:
            collected_constraints.append(c)
            k = (c["column"], c["op"], str(c["value"]))
            weights[k] = w
        else:
            weights[(slot, "NA", "NA")] = w

    collected_constraints = dedup_constraints(collected_constraints)

    base = parse_base_from_sentence(df, base_query)
    full = apply_constraints(df, base, collected_constraints)

    record = {
        "id": i,
        "base_query_sentence": base_query,
        "slots_provided_to_agent": slots,
        "dialogue": history,
        "parsed_base": base,
        "parsed_additional_constraints": collected_constraints,
        "constraint_weights": [
            {"column": p["column"], "op": p["op"], "value": p["value"],
             "weight": float(weights.get((p["column"], p["op"], str(p["value"])), 0.5))}
            for p in collected_constraints
        ],
        "status": None,
        "relaxed_constraint": None,
        "recommended_car": None
    }

    if len(full) > 0:
        cand = full.reset_index(drop=True)
        wmap = {(p["column"], p["op"], str(p["value"])): float(weights.get((p["column"], p["op"], str(p["value"])), 0.5))
                for p in collected_constraints}

        scores = np.zeros(len(cand), dtype=float)
        norms = {}
        for col in ["Year","city mpg","highway MPG","MSRP"]:
            if col in cand.columns:
                norms[col] = normalize_series(cand[col].values)

        for p in collected_constraints:
            k = (p["column"], p["op"], str(p["value"]))
            w = wmap.get(k, 0.25)
            col, op, val = p["column"], p["op"], p["value"]
            if op == "==":
                scores += w
            elif op == ">=":
                if col in norms:
                    scores += w * norms[col]
            elif op == "<=":
                if col in norms:
                    scores += w * (1.0 - norms[col])

        msrp = cand["MSRP"].values.astype(float) if "MSRP" in cand.columns else np.zeros(len(cand))
        best_idx = np.lexsort((msrp, -scores))[0]
        row = cand.iloc[best_idx]

        def gi(k):
            return int(row[k]) if k in cand.columns and pd.notna(row[k]) else None

        record["status"] = "SAT_no_relaxation"
        record["recommended_car"] = {
            "Make": row.get("Make", None),
            "Model": row.get("Model", None),
            "Year": gi("Year"),
            "Engine Fuel Type": row.get("Engine Fuel Type", None),
            "Transmission Type": row.get("Transmission Type", None),
            "Driven_Wheels": row.get("Driven_Wheels", None),
            "Vehicle Size": row.get("Vehicle Size", None),
            "Vehicle Style": row.get("Vehicle Style", None),
            "city mpg": gi("city mpg"),
            "highway MPG": gi("highway MPG"),
            "MSRP": gi("MSRP"),
        }
        outputs.append(record)
        print(f"[{i}] SAT with full query (unexpected for infeasible benchmark). Results={len(full)}")
        continue

    if len(collected_constraints) == 0:
        record["status"] = "UNSAT_no_constraints_collected"
        outputs.append(record)
        print(f"[{i}] UNSAT and no constraints collected.")
        continue

    wmap = {(p["column"], p["op"], str(p["value"])): float(weights.get((p["column"], p["op"], str(p["value"])), 0.5))
            for p in collected_constraints}
    relaxed_key = sorted(wmap.items(), key=lambda kv: kv[1])[0][0]
    relaxed_constraint = None
    for p in collected_constraints:
        if (p["column"], p["op"], str(p["value"])) == relaxed_key:
            relaxed_constraint = {"column": p["column"], "op": p["op"], "value": p["value"]}
            break

    U = df[(df["Make"] == base["Make"]) &
           (df["Transmission Type"] == base["Transmission Type"]) &
           (df["Driven_Wheels"] == base["Driven_Wheels"]) &
           (df["Vehicle Style"] == base["Vehicle Style"])].copy().reset_index(drop=True)

    rec = recommend_after_relaxation(U, collected_constraints, wmap, relaxed_key)

    record["status"] = "SAT_after_relaxation" if rec is not None else "UNSAT_even_after_relaxation"
    record["relaxed_constraint"] = relaxed_constraint
    record["recommended_car"] = rec

    outputs.append(record)
    print(f"[{i}] UNSAT -> relaxed {relaxed_constraint} -> {record['status']}")

with open(AGENT_OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(outputs, f, indent=2, ensure_ascii=False)

print(f"\nWrote outputs to: {AGENT_OUT_PATH}")

# ============================================================
# Simple benchmark comparison (relaxed constraint + recommended car)
# ============================================================
def ckey(c):
    if c is None:
        return None
    return (c.get("column"), c.get("op"), str(c.get("value")))

def car_key(car):
    if not isinstance(car, dict) or not car:
        return None
    return (car.get("Make"), car.get("Model"), str(car.get("Year")), str(car.get("MSRP")))

bench_by_base = {b["base_query_sentence"]: b for b in bench}

relax_matches = 0
car_matches = 0
n = 0

for a in outputs:
    base = a["base_query_sentence"]
    if base not in bench_by_base:
        print(f"\n[SKIP] Not found in bench: {base}")
        continue

    b = bench_by_base[base]
    n += 1

    oracle_relaxed = (
        b.get("unique_repair_constraint")
        or b.get("chosen_relaxation")
        or b.get("relaxed_constraint")
    )
    agent_relaxed = a.get("relaxed_constraint")

    relax_ok = (ckey(oracle_relaxed) == ckey(agent_relaxed))
    relax_matches += int(relax_ok)

    oracle_car = b.get("recommended_car")
    agent_car = a.get("recommended_car")

    # penalty: car match cannot count unless relaxation matches
    car_ok = relax_ok and (car_key(oracle_car) == car_key(agent_car))
    car_matches += int(car_ok)

    print(f"\n=== {n} ===")
    print("Base:", base)
    print("Oracle relaxed:", oracle_relaxed)
    print("Agent  relaxed:", agent_relaxed)
    print("Relax match:", relax_ok)
    print("Oracle car key:", car_key(oracle_car))
    print("Agent  car key:", car_key(agent_car))
    print("Car match:", car_ok)

print("\n=== SUMMARY ===")
print("Compared:", n)
print("Relax matches:", relax_matches, f"({relax_matches/n:.2%})" if n else "")
print("Car matches:", car_matches, f"({car_matches/n:.2%})" if n else "")


In [ ]:
# ============================================================
# Two-LLM simulation (Roleplayer + Agent) for
#          "Formal-Verification-style" infeasibility resolution baseline
#
# Roleplayer: sees ONLY base_query_sentence + persona
# Agent: sees base_query_sentence + list of missing SLOTS (columns only)
# Agent asks clarifications one-by-one, roleplayer answers from persona,
# agent extracts structured constraints (NO weights), then:
#   satisfy-first -> if UNSAT: MUS-like core + minimal-change repair (Hao-style)
# ============================================================

import os, json, re
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from setproctitle import setproctitle
setproctitle("python")

os.environ["CUDA_VISIBLE_DEVICES"] = "4"

# ----------------------------
# Config
# ----------------------------
BENCH_PATH_CANDS = ["./44fix.json", "/mnt/data/queries_10.json"]
DATA_PATH_CANDS  = ["./data.csv", "/mnt/data/data.csv"]

OUT_PATH = "./gamma/gamma_formal44.json"
MODEL_NAME = "google/gemma-2-9b-it"

EXCLUDE = {"Engine HP", "Engine Cylinders", "Market Category", "Popularity", "Number of Doors"}
BASE_COLS = ["Make", "Transmission Type", "Driven_Wheels", "Vehicle Style"]

NUMERIC_COLS = {"Year", "city mpg", "highway MPG", "MSRP"}
CATEGORICAL_COLS = {"Engine Fuel Type", "Vehicle Size", "Model"}
ALLOWED_EXTRA_COLS = list(NUMERIC_COLS | CATEGORICAL_COLS)
ALLOWED_OPS = {"==", ">=", "<="}

MAX_REPAIR_ITERS = 12
MAX_CAT_ALTS = 10

MAX_NEW_TOKENS_Q = 120
MAX_NEW_TOKENS_A = 220
MAX_NEW_TOKENS_PARSE = 240
MAX_NEW_TOKENS_REPAIR = 320

# ----------------------------
# Environment / GPUs
# ----------------------------
# Keep your original GPU selection if desired; safe no-op if unavailable
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2,3")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ----------------------------
# Load bench + data
# ----------------------------
bench_path = next((p for p in BENCH_PATH_CANDS if os.path.exists(p)), None)
if bench_path is None:
    raise FileNotFoundError("Could not find queries_10.json in ./ or /mnt/data/")

data_path = next((p for p in DATA_PATH_CANDS if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError("Could not find data.csv in ./ or /mnt/data/")

with open(bench_path, "r", encoding="utf-8") as f:
    bench = json.load(f)

df0 = pd.read_csv(data_path)
df = df0[[c for c in df0.columns if c not in EXCLUDE]].copy()
if "Engine Fuel Type" in df.columns:
    df["Engine Fuel Type"] = df["Engine Fuel Type"].fillna("Unknown")

# ----------------------------
# Gemma 2 load + robust JSON extraction
# ----------------------------
def load_gemma2(model_name: str = MODEL_NAME):
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    dtype_try = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    try:
        mdl = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=dtype_try,
        )
    except Exception:
        mdl = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        )

    mdl.eval()
    return tok, mdl

tok, mdl = load_gemma2(MODEL_NAME)

def _extract_json_robust(text: str) -> dict:
    t = text.strip()
    if "```" in t:
        parts = t.split("```")
        for i in range(len(parts)-1):
            fence = parts[i].strip().lower()
            payload = parts[i+1].strip()
            if "json" in fence or fence == "":
                lines = payload.splitlines()
                if lines and lines[0].strip().lower() == "json":
                    payload = "\n".join(lines[1:]).strip()
                try:
                    return json.loads(payload)
                except Exception:
                    t = payload
                    break
    start = t.find("{")
    if start == -1:
        raise ValueError("No '{' found in model output.")
    depth = 0
    for i in range(start, len(t)):
        ch = t[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return json.loads(t[start:i+1])
    end = t.rfind("}")
    if end != -1 and end > start:
        return json.loads(t[start:end+1])
    raise ValueError("Failed to parse JSON from model output.")

def _gemma_apply_chat(user_content: str) -> str:
    """
    Gemma IT models in HF generally use roles 'user' and 'model'.
    We inline any system prompt into the user content, then rely on the
    tokenizer chat template; if unavailable, fall back to <start_of_turn>.
    """
    chat = [{"role": "user", "content": user_content}]
    try:
        return tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    except Exception:
        bos = tok.bos_token or "<bos>"
        return f"{bos}<start_of_turn>user\n{user_content}<end_of_turn>\n<start_of_turn>model\n"

@torch.inference_mode()
def gemma_gen_text(system_prompt: str, user_prompt: str, max_new_tokens: int = 200) -> str:
    combined = (system_prompt or "").strip()
    if combined:
        combined = combined + "\n\n" + (user_prompt or "").strip()
    else:
        combined = (user_prompt or "").strip()

    prompt_text = _gemma_apply_chat(combined)
    inputs = tok(prompt_text, return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(mdl.device) for k, v in inputs.items()}
    if "attention_mask" not in inputs or inputs["attention_mask"] is None:
        inputs["attention_mask"] = (inputs["input_ids"] != tok.pad_token_id).long()

    prompt_len = inputs["input_ids"].shape[-1]
    out = mdl.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
    )
    gen_ids = out[0][prompt_len:]
    return tok.decode(gen_ids, skip_special_tokens=True).strip()

@torch.inference_mode()
def gemma_gen_json(system_prompt: str, user_prompt: str, max_new_tokens: int = 260, retries: int = 3) -> dict:
    strict_suffixes = [
        "",
        "\n\nIMPORTANT: Output MUST start with '{' and end with '}' and contain NOTHING else.",
        "\n\nFINAL WARNING: ONLY output JSON. No markdown. No code fences. No commentary."
    ]
    last_text, last_err = None, None
    for attempt in range(retries):
        sys = (system_prompt or "") + strict_suffixes[min(attempt, len(strict_suffixes)-1)]
        txt = gemma_gen_text(sys, user_prompt, max_new_tokens=max_new_tokens)
        last_text = txt
        try:
            return _extract_json_robust(txt)
        except Exception as e:
            last_err = e
            continue
    raise ValueError(f"Could not parse JSON after {retries} tries. Last error: {last_err}\n\nLast model output:\n{last_text}")

# ----------------------------
# Parsing / sanitization helpers
# ----------------------------
def coerce_int(v) -> Optional[int]:
    if v is None:
        return None
    if isinstance(v, (int, np.integer)):
        return int(v)
    if isinstance(v, float) and np.isfinite(v):
        return int(round(v))
    s = str(v).strip()
    m = re.search(r"-?\d+", s)
    if not m:
        return None
    try:
        return int(m.group(0))
    except Exception:
        return None

def sanitize_constraint(p: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    if not isinstance(p, dict):
        return None
    col, op, val = p.get("column"), p.get("op"), p.get("value")
    if col not in ALLOWED_EXTRA_COLS or op not in ALLOWED_OPS:
        return None
    if col in NUMERIC_COLS:
        if op in (">=", "<=", "=="):
            iv = coerce_int(val)
            if iv is None:
                return None
            return {"column": col, "op": op, "value": iv}
        return None
    if col in CATEGORICAL_COLS:
        if op != "==":
            return None
        sval = str(val).strip()
        if sval == "":
            return None
        return {"column": col, "op": "==", "value": sval}
    return None

def dedup_constraints(extras: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    out = []
    for p in extras:
        k = (p["column"], p["op"], str(p["value"]))
        if k in seen:
            continue
        seen.add(k)
        out.append(p)
    return out

# ----------------------------
# Base parsing from base sentence (deterministic vocab match)
# ----------------------------
def parse_base_from_sentence(df: pd.DataFrame, base_sentence: str) -> Dict[str, str]:
    s = base_sentence.lower()
    def match(col: str) -> str:
        vocab = sorted(df[col].dropna().astype(str).unique().tolist(), key=len, reverse=True)
        for v in vocab:
            if v.lower() in s:
                return v
        raise ValueError(f"Could not parse base field {col} from: {base_sentence}")
    return {
        "Make": match("Make"),
        "Vehicle Style": match("Vehicle Style"),
        "Transmission Type": match("Transmission Type"),
        "Driven_Wheels": match("Driven_Wheels"),
    }

# ----------------------------
# DB "solver"
# ----------------------------
def apply_constraints(df: pd.DataFrame, base: Dict[str, str], extras: List[Dict[str, Any]]) -> pd.DataFrame:
    m = np.ones(len(df), dtype=bool)
    for c in BASE_COLS:
        m &= (df[c].values == base[c])
    for p in extras:
        col, op, val = p["column"], p["op"], p["value"]
        if col not in df.columns:
            return df.iloc[0:0].copy()
        if op == "==":
            m &= (df[col].astype(str).values == str(val))
        elif op == ">=":
            m &= (df[col].values >= int(val))
        elif op == "<=":
            m &= (df[col].values <= int(val))
    return df[m].copy()

def is_sat(df: pd.DataFrame, base: Dict[str, str], extras: List[Dict[str, Any]]) -> bool:
    return len(apply_constraints(df, base, extras)) > 0

def base_slice(df: pd.DataFrame, base: Dict[str, str]) -> pd.DataFrame:
    m = np.ones(len(df), dtype=bool)
    for c in BASE_COLS:
        m &= (df[c].values == base[c])
    return df[m].copy()

# ----------------------------
# MUS-like unsat core (deletion)
# ----------------------------
def mus_deletion_core(df: pd.DataFrame, base: Dict[str, str], extras: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    core = extras[:]
    if is_sat(df, base, core):
        return []
    changed = True
    while changed:
        changed = False
        i = 0
        while i < len(core):
            test = core[:i] + core[i+1:]
            if not is_sat(df, base, test):
                core = test
                changed = True
            else:
                i += 1
    return core

# ----------------------------
# Minimal-change repair candidates (Hao-style)
# ----------------------------
def feasible_under_others(dfU: pd.DataFrame, others: List[Dict[str, Any]]) -> pd.DataFrame:
    m = np.ones(len(dfU), dtype=bool)
    for p in others:
        col, op, val = p["column"], p["op"], p["value"]
        if op == "==":
            m &= (dfU[col].astype(str).values == str(val))
        elif op == ">=":
            m &= (dfU[col].values >= int(val))
        elif op == "<=":
            m &= (dfU[col].values <= int(val))
    return dfU[m].copy()

def propose_minimal_repairs(df: pd.DataFrame, base: Dict[str, str],
                            extras: List[Dict[str, Any]], core: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    U = base_slice(df, base)
    candidates = []
    for target in core:
        others = [p for p in extras if p is not target]
        Fe = feasible_under_others(U, others)

        col, op, val = target["column"], target["op"], target["value"]

        # Drop: always possible (cost=1.0)
        candidates.append({
            "action": "drop",
            "from": target,
            "to": None,
            "cost": 1.0,
            "rationale": f"Drop {col} {op} {val}"
        })

        if len(Fe) == 0 or col not in Fe.columns:
            continue

        if op == "==" and col in CATEGORICAL_COLS:
            vc = Fe[col].astype(str).value_counts()
            alts = [x for x in vc.index.tolist() if x != str(val)][:MAX_CAT_ALTS]
            for rank, alt in enumerate(alts):
                candidates.append({
                    "action": "replace",
                    "from": target,
                    "to": {"column": col, "op": "==", "value": alt},
                    "cost": 0.25 + 0.05 * rank,
                    "rationale": f"Change {col} to common feasible value {alt}"
                })

        elif op == ">=" and col in NUMERIC_COLS:
            series = np.sort(Fe[col].dropna().astype(float).unique())
            if len(series) == 0:
                continue
            vals = series[series <= float(val)]
            new_t = float(series.max()) if len(vals) == 0 else float(vals.max())
            if new_t != float(val):
                rngv = max(1.0, float(series.max() - series.min()))
                cost = abs(float(val) - new_t) / rngv
                candidates.append({
                    "action": "replace",
                    "from": target,
                    "to": {"column": col, "op": ">=", "value": int(round(new_t))},
                    "cost": float(cost),
                    "rationale": f"Lower {col} to nearest feasible threshold {int(round(new_t))}"
                })

        elif op == "<=" and col in NUMERIC_COLS:
            series = np.sort(Fe[col].dropna().astype(float).unique())
            if len(series) == 0:
                continue
            vals = series[series >= float(val)]
            new_t = float(series.max()) if len(vals) == 0 else float(vals.min())
            if new_t != float(val):
                rngv = max(1.0, float(series.max() - series.min()))
                cost = abs(new_t - float(val)) / rngv
                candidates.append({
                    "action": "replace",
                    "from": target,
                    "to": {"column": col, "op": "<=", "value": int(round(new_t))},
                    "cost": float(cost),
                    "rationale": f"Raise {col} to nearest feasible cap {int(round(new_t))}"
                })
    candidates.sort(key=lambda x: x["cost"])
    return candidates

def constraints_equal(a: Dict[str, Any], b: Dict[str, Any]) -> bool:
    return a["column"] == b["column"] and a["op"] == b["op"] and str(a["value"]) == str(b["value"])

def apply_repair(extras: List[Dict[str, Any]], repair: Dict[str, Any]) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    frm = repair["from"]
    action = repair["action"]
    to = repair.get("to", None)

    new_extras = []
    replaced = False
    for p in extras:
        if (not replaced) and constraints_equal(p, frm):
            replaced = True
            if action == "drop":
                continue
            elif action == "replace":
                new_extras.append(to)
            else:
                raise ValueError(f"Unknown action: {action}")
        else:
            new_extras.append(p)

    if not replaced and action == "replace" and to is not None:
        new_extras.append(to)

    return dedup_constraints(new_extras), {"action": action, "from": frm, "to": to}

# ----------------------------
# Recommendation
# ----------------------------
def recommend_car(feasible: pd.DataFrame) -> Optional[Dict[str, Any]]:
    if len(feasible) == 0:
        return None
    by, asc = [], []
    if "MSRP" in feasible.columns:
        by.append("MSRP"); asc.append(True)
    if "Year" in feasible.columns:
        by.append("Year"); asc.append(False)
    if "highway MPG" in feasible.columns:
        by.append("highway MPG"); asc.append(False)
    row = feasible.sort_values(by=by, ascending=asc).iloc[0] if by else feasible.iloc[0]

    def gi(k):
        return int(row[k]) if k in feasible.columns and pd.notna(row[k]) else None

    return {
        "Make": row.get("Make", None),
        "Model": row.get("Model", None),
        "Year": gi("Year"),
        "Engine Fuel Type": row.get("Engine Fuel Type", None),
        "Transmission Type": row.get("Transmission Type", None),
        "Driven_Wheels": row.get("Driven_Wheels", None),
        "Vehicle Size": row.get("Vehicle Size", None),
        "Vehicle Style": row.get("Vehicle Style", None),
        "city mpg": gi("city mpg"),
        "highway MPG": gi("highway MPG"),
        "MSRP": gi("MSRP"),
    }

# ----------------------------
# Two-LLM conversation simulation (clarify -> fill slots)
# ----------------------------
ROLEPLAYER_SYS = (
    "You are ROLEPLAYER_USER. You ONLY know the provided persona and the base query. "
    "Answer the assistant's question using ONLY information that is stated or clearly implied in the persona. "
    "IMPORTANT: When the assistant is clarifying a constraint, you should be helpful and specific: "
    "if the persona provides ANY relevant value (e.g., a specific model name, a year minimum, a budget cap), "
    "you MUST state that value explicitly in your answer. "
    "Do NOT say 'no preference' or omit the value if the persona mentions it. "
    "Only say you have no preference if the persona truly contains no information about that attribute. "
    "If the assistant's question is a bit vague, still provide the most relevant constraint value(s) from the persona. "
    "Do not invent details that are not in the persona."
)

AGENT_ASK_SYS = (
    "You are CLARIFYING_AGENT. Ask ONE short question to learn the user's requirement for the given slot/field. "
    "Do not propose relaxations. Do not ask multiple questions at once."
)

AGENT_EXTRACT_SYS = (
    "You are a strict JSON generator. Extract the requirement for the given slot from the user's answer. "
    "Output ONLY JSON."
)

def agent_question(slot: str, base_query: str, history: List[Dict[str,str]]) -> str:
    hist_txt = "\n".join([f"{m['role'].upper()}: {m['content']}" for m in history[-6:]])
    user_prompt = f"""
Base query: {base_query}

Conversation so far:
{hist_txt}

Slot to clarify: {slot}

Ask ONE question to determine the user's requirement for {slot}. Keep it concise.
""".strip()
    return gemma_gen_text(AGENT_ASK_SYS, user_prompt, max_new_tokens=MAX_NEW_TOKENS_Q)

def roleplayer_answer(persona: str, base_query: str, question: str) -> str:
    user_prompt = f"""
Base query the user started with:
{base_query}

Persona (all you know):
{persona}

Assistant asked:
{question}

Answer as the user, using only persona information.
""".strip()
    return gemma_gen_text(ROLEPLAYER_SYS, user_prompt, max_new_tokens=MAX_NEW_TOKENS_A)

def extract_constraint_from_answer(slot: str, user_answer: str) -> Optional[Dict[str,Any]]:
    user_prompt = f"""
Slot: {slot}
User answer: {user_answer}

Return JSON:
{{
  "constraint": null OR {{"column": "{slot}", "op": "=="|">="|"<=", "value": string|number}}
}}

Rules:
- If user has no preference / doesn't care, output constraint:null.
- If slot is numeric (Year, city mpg, highway MPG, MSRP):
  - Use >= for minimum requirements, <= for maximum cap, == only if explicitly exact.
  - value MUST be an integer.
- If slot is categorical (Engine Fuel Type, Vehicle Size, Model):
  - Use op == and value as string.
- Output ONLY JSON.
""".strip()
    obj = gemma_gen_json(AGENT_EXTRACT_SYS, user_prompt, max_new_tokens=MAX_NEW_TOKENS_PARSE, retries=3)
    c = obj.get("constraint", None)
    if c is None:
        return None
    c["column"] = slot
    return sanitize_constraint(c)

# ----------------------------
# Repair selection prompt (LLM picks one repair)
# ----------------------------
SYSTEM_REPAIR = (
    "You are a strict JSON generator performing minimal-change UNSAT repair. Output ONLY JSON. "
    "You must choose ONE repair."
)

def build_repair_prompt(base: Dict[str, str], extras: List[Dict[str, Any]],
                        core: List[Dict[str, Any]], candidates: List[Dict[str, Any]]) -> str:
    top = candidates[:20]
    return f"""
Current query is UNSAT (empty results).

Base:
{json.dumps(base, ensure_ascii=False)}

Additional constraints:
{json.dumps(extras, ensure_ascii=False)}

Unsat core:
{json.dumps(core, ensure_ascii=False)}

Candidate minimal-change repairs (lower cost = smaller change):
{json.dumps(top, ensure_ascii=False)}

Pick ONE repair (prefer smallest cost that is likely to make results non-empty).
Return JSON:
{{
  "repair": {{
    "action": "drop" | "replace",
    "from": {{"column":..., "op":..., "value":...}},
    "to": null OR {{"column":..., "op":..., "value":...}}
  }}
}}
Output ONLY JSON.
""".strip()

# ----------------------------
# Run: for each benchmark item, do dialogue -> full query -> repair loop
# ----------------------------
outputs = []

for qi, b in enumerate(bench, 1):
    base_query = b["base_query_sentence"]
    persona = b.get("persona", "")

    slots = sorted({c["column"] for c in b.get("additional_constraints", [])})
    base = parse_base_from_sentence(df, base_query)

    history = [{"role":"user","content": base_query}]
    extras: List[Dict[str,Any]] = []

    # Clarify each slot one-by-one
    for slot in slots:
        qtxt = agent_question(slot, base_query, history)
        history.append({"role":"assistant","content": qtxt})

        atxt = roleplayer_answer(persona, base_query, qtxt)
        history.append({"role":"user","content": atxt})

        c = extract_constraint_from_answer(slot, atxt)
        if c is not None:
            extras.append(c)

    extras = dedup_constraints(extras)

    record = {
        "id": qi,
        "base_query_sentence": base_query,
        "slots_provided_to_agent": slots,
        "dialogue": history,
        "parsed_base": base,
        "parsed_additional_constraints": extras,
        "status": None,
        "iterations": [],
        "relaxed_constraint": None,
        "recommended_car": None
    }

    # satisfy-first
    feasible = apply_constraints(df, base, extras)
    if len(feasible) > 0:
        record["status"] = "SAT_no_relaxation"
        record["recommended_car"] = recommend_car(feasible)
        outputs.append(record)
        print(f"[{qi}] SAT with full query. Results={len(feasible)} (extras={len(extras)}/{len(slots)})")
        continue

    # UNSAT -> formal-verif style loop (core -> candidates -> LLM picks repair)
    cur = extras[:]
    first_relax = None

    for it in range(MAX_REPAIR_ITERS):
        core = mus_deletion_core(df, base, cur)
        candidates = propose_minimal_repairs(df, base, cur, core)

        if not candidates:
            target = core[0] if core else (cur[-1] if cur else None)
            if target is None:
                break
            chosen = {"repair": {"action": "drop", "from": target, "to": None}}
        else:
            chosen = gemma_gen_json(
                SYSTEM_REPAIR,
                build_repair_prompt(base, cur, core, candidates),
                max_new_tokens=MAX_NEW_TOKENS_REPAIR,
                retries=3
            )

        repair = chosen["repair"]
        rep_from = sanitize_constraint(repair.get("from", {}))
        rep_to = None if repair.get("to") is None else sanitize_constraint(repair.get("to", {}))
        rep_action = repair.get("action")

        if rep_from is None or rep_action not in ("drop", "replace"):
            target = core[0] if core else (cur[-1] if cur else None)
            if target is None:
                break
            rep_action, rep_from, rep_to = "drop", target, None

        safe_repair = {"action": rep_action, "from": rep_from, "to": rep_to}
        new_cur, relaxed = apply_repair(cur, safe_repair)
        feasible = apply_constraints(df, base, new_cur)

        record["iterations"].append({
            "iter": it,
            "unsat_core": core,
            "num_candidates_presented": int(min(20, len(candidates))),
            "chosen_repair": relaxed,
            "num_results_after": int(len(feasible)),
        })

        if first_relax is None:
            first_relax = relaxed

        cur = new_cur
        if len(feasible) > 0:
            record["status"] = "SAT_after_relaxation"
            record["relaxed_constraint"] = first_relax
            record["recommended_car"] = recommend_car(feasible)
            print(f"[{qi}] SAT after iter {it}. Results={len(feasible)} (extras={len(extras)}/{len(slots)})")
            break

    if record["status"] is None:
        record["status"] = "UNSAT_timeout"
        record["relaxed_constraint"] = first_relax
        record["recommended_car"] = None
        print(f"[{qi}] UNSAT after {MAX_REPAIR_ITERS} iters.")

    outputs.append(record)

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(outputs, f, indent=2, ensure_ascii=False)

print(f"\nWrote {len(outputs)} outputs to: {OUT_PATH}")
print("Example record:\n", json.dumps(outputs[0], indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# Two-LLM simulation (Roleplayer + Agent) + OPTION A One-shot global weighting pass
#
# Notes on Gemma-2-9b-it usage (per HF model card conventions):
# - Chat template typically uses roles: "user" and "model" (no dedicated "system" role).
# - We INLINE the "system prompt" into the user content.
# - We use tokenizer.apply_chat_template(..., add_generation_prompt=True) when available,
#   else fall back to the <start_of_turn> format.
# ============================================================

import os, json, re
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from setproctitle import setproctitle
setproctitle("python")

# keep original GPU constraint if desired
os.environ["CUDA_VISIBLE_DEVICES"] = "4"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ----------------------------
# Paths / config
# ----------------------------
BENCH_PATH_CANDS = ["./44fix.json", "/mnt/data/queries_10.json", "/mnt/data/44fix.json"]
DATA_PATH_CANDS  = ["./data.csv", "/mnt/data/data.csv"]

AGENT_OUT_PATH = "./gamma/gamma_global44.json"

MODEL_NAME = "google/gemma-2-9b-it"
SEED = 11
rng = np.random.default_rng(SEED)

EXCLUDE = {"Engine HP", "Engine Cylinders", "Market Category", "Popularity", "Number of Doors"}
BASE_COLS = ["Make", "Transmission Type", "Driven_Wheels", "Vehicle Style"]

NUMERIC_COLS = {"Year", "city mpg", "highway MPG", "MSRP"}
CATEGORICAL_COLS = {"Engine Fuel Type", "Vehicle Size", "Model"}
ALLOWED_EXTRA_COLS = list(NUMERIC_COLS | CATEGORICAL_COLS)
ALLOWED_OPS = {"==", ">=", "<="}

MAX_NEW_TOKENS_Q = 140
MAX_NEW_TOKENS_A = 220
MAX_NEW_TOKENS_PARSE = 240
MAX_NEW_TOKENS_WEIGHT = 280

# one retry ask if extraction fails but answer is not "no preference"
MAX_REASKS_PER_SLOT = 1

# ----------------------------
# Load benchmark + data
# ----------------------------
bench_path = next((p for p in BENCH_PATH_CANDS if os.path.exists(p)), None)
if bench_path is None:
    raise FileNotFoundError("Could not find benchmark json in ./ or /mnt/data/")

data_path = next((p for p in DATA_PATH_CANDS if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError("Could not find data.csv in ./ or /mnt/data/")

with open(bench_path, "r", encoding="utf-8") as f:
    bench = json.load(f)

df0 = pd.read_csv(data_path)
df = df0[[c for c in df0.columns if c not in EXCLUDE]].copy()
if "Engine Fuel Type" in df.columns:
    df["Engine Fuel Type"] = df["Engine Fuel Type"].fillna("Unknown")

# ----------------------------
# Canonicalization helpers
# ----------------------------
def _build_vocab_maps(df: pd.DataFrame, cols: List[str]):
    vocab_list = {}
    vocab_lower_to_orig = {}
    for c in cols:
        if c not in df.columns:
            continue
        vals = df[c].dropna().astype(str).unique().tolist()
        vocab_list[c] = vals
        m = {}
        for v in vals:
            vl = v.strip().lower()
            if vl and vl not in m:
                m[vl] = v
        vocab_lower_to_orig[c] = m
    return vocab_list, vocab_lower_to_orig

VOCAB_LIST, VOCAB_LOWER_TO_ORIG = _build_vocab_maps(df, list(CATEGORICAL_COLS))
MAKE_SET_LOWER = set(df["Make"].dropna().astype(str).str.strip().str.lower().unique().tolist()) if "Make" in df.columns else set()

def canonicalize_categorical(col: str, raw: str) -> str:
    s = (raw or "").strip()
    if s == "" or col not in VOCAB_LOWER_TO_ORIG:
        return s

    low = s.lower()
    m = VOCAB_LOWER_TO_ORIG[col]
    if low in m:
        return m[low]

    if col == "Model":
        toks = re.split(r"\s+", s)
        if toks:
            t0 = toks[0].strip().lower()
            if t0 in MAKE_SET_LOWER and len(toks) >= 2:
                s2 = " ".join(toks[1:]).strip()
                if s2.lower() in m:
                    return m[s2.lower()]

    def _clean(x: str) -> str:
        x = x.lower()
        x = re.sub(r"[^a-z0-9\s\-\/]", " ", x)
        x = re.sub(r"\b(the|a|an|model|trim|edition)\b", " ", x)
        x = re.sub(r"\s+", " ", x).strip()
        return x

    c_low = _clean(s)
    best = None
    best_len = -1
    for v in VOCAB_LIST.get(col, []):
        vl = _clean(str(v))
        if not vl:
            continue
        if vl in c_low or c_low in vl:
            if len(vl) > best_len:
                best = v
                best_len = len(vl)
    return str(best) if best is not None else s

# ----------------------------
# Gemma 2 load + robust JSON extraction
# ----------------------------
def load_gemma2(model_name: str = MODEL_NAME):
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    dtype_try = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    try:
        mdl = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=dtype_try,
        )
    except Exception:
        mdl = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        )
    mdl.eval()
    return tok, mdl

tok, mdl = load_gemma2(MODEL_NAME)

def _extract_json_robust(text: str) -> dict:
    t = text.strip()
    if "```" in t:
        parts = t.split("```")
        for i in range(len(parts)-1):
            fence = parts[i].strip().lower()
            payload = parts[i+1].strip()
            if "json" in fence or fence == "":
                lines = payload.splitlines()
                if lines and lines[0].strip().lower() == "json":
                    payload = "\n".join(lines[1:]).strip()
                try:
                    return json.loads(payload)
                except Exception:
                    t = payload
                    break
    start = t.find("{")
    if start == -1:
        raise ValueError("No '{' found in model output.")
    depth = 0
    for i in range(start, len(t)):
        ch = t[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return json.loads(t[start:i+1])
    end = t.rfind("}")
    if end != -1 and end > start:
        return json.loads(t[start:end+1])
    raise ValueError("Failed to parse JSON object from model output.")

def _gemma_apply_chat(user_content: str) -> str:
    """
    Gemma IT models generally use roles 'user' and 'model'.
    We inline 'system' instructions into user_content and call apply_chat_template.
    """
    chat = [{"role": "user", "content": user_content}]
    try:
        return tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    except Exception:
        bos = tok.bos_token or "<bos>"
        return f"{bos}<start_of_turn>user\n{user_content}<end_of_turn>\n<start_of_turn>model\n"

@torch.inference_mode()
def gemma_gen_text(system_prompt: str, user_prompt: str, max_new_tokens: int = 200) -> str:
    combined = (system_prompt or "").strip()
    if combined:
        combined = combined + "\n\n" + (user_prompt or "").strip()
    else:
        combined = (user_prompt or "").strip()

    prompt_text = _gemma_apply_chat(combined)
    inputs = tok(prompt_text, return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(mdl.device) for k, v in inputs.items()}
    if "attention_mask" not in inputs or inputs["attention_mask"] is None:
        inputs["attention_mask"] = (inputs["input_ids"] != tok.pad_token_id).long()

    prompt_len = inputs["input_ids"].shape[-1]
    out = mdl.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
    )
    gen_ids = out[0][prompt_len:]
    return tok.decode(gen_ids, skip_special_tokens=True).strip()

@torch.inference_mode()
def gemma_gen_json(system_prompt: str, user_prompt: str, max_new_tokens: int = 260, retries: int = 3) -> dict:
    strict_suffixes = [
        "",
        "\n\nIMPORTANT: Output MUST start with '{' and end with '}' and contain NOTHING else.",
        "\n\nFINAL WARNING: ONLY output JSON. No markdown. No code fences. No commentary."
    ]
    last_text, last_err = None, None
    for attempt in range(retries):
        sys = (system_prompt or "") + strict_suffixes[min(attempt, len(strict_suffixes)-1)]
        txt = gemma_gen_text(sys, user_prompt, max_new_tokens=max_new_tokens)
        last_text = txt
        try:
            return _extract_json_robust(txt)
        except Exception as e:
            last_err = e
    raise ValueError(f"Could not parse JSON after {retries} tries. Last error: {last_err}\n\nLast model output:\n{last_text}")

# ----------------------------
# Sanitizers + utilities
# ----------------------------
NO_PREF_PAT = re.compile(r"\b(no preference|don't care|do not care|no pref|any is fine|whatever|doesn['’]?t matter)\b", re.I)

def coerce_int(v) -> Optional[int]:
    if v is None:
        return None
    if isinstance(v, (int, np.integer)):
        return int(v)
    if isinstance(v, float) and np.isfinite(v):
        return int(round(v))
    s = str(v).strip()
    m = re.search(r"-?\d+", s)
    if not m:
        return None
    try:
        return int(m.group(0))
    except Exception:
        return None

def sanitize_constraint(p: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    if not isinstance(p, dict):
        return None
    col, op, val = p.get("column"), p.get("op"), p.get("value")
    if col not in ALLOWED_EXTRA_COLS or op not in ALLOWED_OPS:
        return None

    if col in NUMERIC_COLS:
        iv = coerce_int(val)
        if iv is None:
            return None
        return {"column": col, "op": op, "value": iv}

    if col in CATEGORICAL_COLS:
        if op != "==":
            return None
        sval = str(val).strip()
        if sval == "":
            return None
        sval = canonicalize_categorical(col, sval)
        if sval == "":
            return None
        return {"column": col, "op": "==", "value": sval}

    return None

def dedup_constraints(extras: List[Dict[str,Any]]) -> List[Dict[str,Any]]:
    seen = set()
    out = []
    for p in extras:
        k = (p["column"], p["op"], str(p["value"]))
        if k in seen:
            continue
        seen.add(k)
        out.append(p)
    return out

# ----------------------------
# Solver filtering
# ----------------------------
def apply_constraints(df: pd.DataFrame, base: Dict[str,str], extras: List[Dict[str,Any]]) -> pd.DataFrame:
    m = np.ones(len(df), dtype=bool)
    for c in BASE_COLS:
        m &= (df[c].values == base[c])
    for p in extras:
        col, op, val = p["column"], p["op"], p["value"]
        if col not in df.columns:
            return df.iloc[0:0].copy()
        if op == "==":
            m &= (df[col].astype(str).values == str(val))
        elif op == ">=":
            m &= (df[col].values >= int(val))
        elif op == "<=":
            m &= (df[col].values <= int(val))
        else:
            return df.iloc[0:0].copy()
    return df[m].copy()

# ----------------------------
# Option A: One-shot global weighting pass AFTER dialogue
# ----------------------------
WEIGHT_SYS = (
    "You assign RELATIVE importance weights to user constraints based on the conversation. "
    "Return nonnegative weights that SUM TO 1. "
    "Higher for must-have/can't compromise; lower for flexible/no preference. "
    "Output ONLY JSON."
)

def global_weighting_pass(history: List[Dict[str,str]], constraints: List[Dict[str,Any]]) -> Dict[Tuple[str,str,str], float]:
    if len(constraints) == 0:
        return {}
    if len(constraints) == 1:
        p = constraints[0]
        return {(p["column"], p["op"], str(p["value"])): 1.0}

    transcript = "\n".join([f"{m['role'].upper()}: {m['content']}" for m in history])
    items = [{"id": i, "column": p["column"], "op": p["op"], "value": p["value"]} for i, p in enumerate(constraints)]

    user_prompt = f"""
Conversation transcript:
{transcript}

Constraints (fixed; do not change):
{json.dumps(items, ensure_ascii=False, indent=2)}

Return JSON:
{{
  "weights": [
    {{"id": 0, "weight": 0.25}},
    ...
  ]
}}
Rules:
- weights >= 0
- SUM(weights) = 1
""".strip()

    obj = gemma_gen_json(WEIGHT_SYS, user_prompt, max_new_tokens=MAX_NEW_TOKENS_WEIGHT, retries=3)
    wlist = obj.get("weights", None)

    if not isinstance(wlist, list) or len(wlist) != len(items):
        uni = 1.0 / len(items)
        return {(p["column"], p["op"], str(p["value"])): uni for p in constraints}

    tmp = {}
    for w in wlist:
        try:
            i = int(w.get("id"))
            val = float(w.get("weight"))
            if i < 0 or i >= len(items) or not np.isfinite(val) or val < 0:
                continue
            tmp[i] = val
        except Exception:
            pass

    if len(tmp) != len(items):
        uni = 1.0 / len(items)
        return {(p["column"], p["op"], str(p["value"])): uni for p in constraints}

    arr = np.array([tmp[i] for i in range(len(items))], dtype=float)
    s = float(arr.sum())
    if not np.isfinite(s) or s <= 0:
        arr[:] = 1.0
        s = float(arr.sum())
    arr /= s

    return {(p["column"], p["op"], str(p["value"])): float(arr[i]) for i, p in enumerate(constraints)}

# ----------------------------
# Soft scoring helpers
# ----------------------------
def normalize_series(x: np.ndarray) -> np.ndarray:
    xmin = float(np.min(x))
    xmax = float(np.max(x))
    if xmax - xmin < 1e-9:
        return np.zeros_like(x, dtype=float)
    return (x - xmin) / (xmax - xmin)

def score_candidates_soft(U: pd.DataFrame, cand: pd.DataFrame, constraints: List[Dict[str,Any]],
                          weights: Dict[Tuple[str,str,str], float]) -> np.ndarray:
    if len(cand) == 0:
        return np.array([], dtype=float)

    scores = np.zeros(len(cand), dtype=float)
    norms = {}
    for col in ["Year","city mpg","highway MPG","MSRP"]:
        if col in U.columns:
            norms[col] = normalize_series(U[col].values)

    idx = cand.index.values
    for p in constraints:
        k = (p["column"], p["op"], str(p["value"]))
        w = float(weights.get(k, 0.0))
        col, op, val = p["column"], p["op"], p["value"]

        if op == "==":
            scores += w * (cand[col].astype(str).values == str(val)).astype(float)
        elif op == ">=":
            scores += w * (norms[col][idx] if col in norms else (cand[col].values >= int(val)).astype(float))
        elif op == "<=":
            scores += w * ((1.0 - norms[col][idx]) if col in norms else (cand[col].values <= int(val)).astype(float))

    return scores

def pick_best_row(cand: pd.DataFrame, scores: np.ndarray) -> pd.DataFrame:
    if len(cand) <= 1:
        return cand.reset_index(drop=True)
    msrp = cand["MSRP"].values.astype(float) if "MSRP" in cand.columns else np.zeros(len(cand))
    best_idx = np.lexsort((msrp, -scores))[0]
    return cand.iloc[[best_idx]].reset_index(drop=True)

def row_to_car(row: pd.Series) -> Dict[str,Any]:
    def gi(k):
        return int(row[k]) if k in row.index and pd.notna(row[k]) else None
    return {
        "Make": row.get("Make", None),
        "Model": row.get("Model", None),
        "Year": gi("Year"),
        "Engine Fuel Type": row.get("Engine Fuel Type", None),
        "Transmission Type": row.get("Transmission Type", None),
        "Driven_Wheels": row.get("Driven_Wheels", None),
        "Vehicle Size": row.get("Vehicle Size", None),
        "Vehicle Style": row.get("Vehicle Style", None),
        "city mpg": gi("city mpg"),
        "highway MPG": gi("highway MPG"),
        "MSRP": gi("MSRP"),
    }

# ----------------------------
# Roleplayer + Agent prompts (natural, anti-invention)
# ----------------------------
ROLEPLAYER_SYS = (
    "You are ROLEPLAYER_USER. You ONLY know the provided persona and the base query.\n"
    "Answer the assistant's question using ONLY information in the persona.\n"
    "If the persona does not specify a preference/value for that attribute, explicitly say you have no preference.\n"
    "Do NOT invent values. If unsure, say 'no preference'.\n"
    "Also state importance naturally (must-have / strongly prefer / nice-to-have / flexible / no preference)."
)

AGENT_ASK_SYS = (
    "You are CLARIFYING_AGENT. Ask ONE short question to learn the user's requirement for the given slot/field.\n"
    "Ask for: (1) the specific value (or 'no preference'), and (2) how important it is (must-have/strongly prefer/nice-to-have/flexible).\n"
    "Do not propose relaxations. Do not ask multiple questions."
)

AGENT_EXTRACT_ONLY_SYS = (
    "You are a strict JSON generator. Extract the constraint value from the user's answer for the given slot.\n"
    "Output ONLY JSON."
)

def agent_question(slot: str, base_query: str, history: List[Dict[str,str]]) -> str:
    hist_txt = "\n".join([f"{m['role'].upper()}: {m['content']}" for m in history[-6:]])
    user_prompt = f"""
Base query: {base_query}

Conversation so far:
{hist_txt}

Slot to clarify: {slot}

Ask ONE concise question for {slot}. Require a specific value (or 'no preference') + importance.
""".strip()
    return gemma_gen_text(AGENT_ASK_SYS, user_prompt, max_new_tokens=MAX_NEW_TOKENS_Q)

def roleplayer_answer(persona: str, base_query: str, question: str) -> str:
    user_prompt = f"""
Base query:
{base_query}

Persona (all you know):
{persona}

Assistant asked:
{question}

Answer as the user, using only persona information.
""".strip()
    return gemma_gen_text(ROLEPLAYER_SYS, user_prompt, max_new_tokens=MAX_NEW_TOKENS_A)

# -------- extractor fallback heuristics (non-cheating) --------
def _best_vocab_match(col: str, text: str) -> Optional[str]:
    if col not in VOCAB_LIST:
        return None
    t = (text or "").lower()

    def _clean(x: str) -> str:
        x = x.lower()
        x = re.sub(r"[^a-z0-9\s\-\/]", " ", x)
        x = re.sub(r"\s+", " ", x).strip()
        return x

    ct = _clean(t)
    best = None
    best_len = -1
    for v in VOCAB_LIST[col]:
        vv = _clean(str(v))
        if not vv:
            continue
        if vv in ct and len(vv) > best_len:
            best = str(v)
            best_len = len(vv)
    return best

def _heuristic_numeric(slot: str, text: str) -> Optional[Dict[str,Any]]:
    m = re.search(r"(-?\d+)", text or "")
    if not m:
        return None
    iv = coerce_int(m.group(1))
    if iv is None:
        return None

    tl = (text or "").lower()
    op = "=="
    if slot == "MSRP":
        if re.search(r"\b(under|below|at most|max(imum)?|budget|no more than|<=)\b", tl):
            op = "<="
        elif re.search(r"\b(at least|min(imum)?|>=|more than)\b", tl):
            op = ">="
        else:
            op = "<="
    else:
        if re.search(r"\b(at least|min(imum)?|>=|no less than)\b", tl):
            op = ">="
        elif re.search(r"\b(under|below|at most|max(imum)?|<=|no more than)\b", tl):
            op = "<="
        elif re.search(r"\b(exact(ly)?|only)\b", tl):
            op = "=="
        else:
            op = ">="
    return {"column": slot, "op": op, "value": iv}

def extract_constraint(slot: str, user_answer: str) -> Optional[Dict[str,Any]]:
    if NO_PREF_PAT.search(user_answer or ""):
        return None

    user_prompt = f"""
Slot: {slot}
User answer: {user_answer}

Return JSON:
{{
  "constraint": null OR {{"column": "{slot}", "op": "=="|">="|"<="| "==", "value": string|number}}
}}

Rules:
- If user has no preference / doesn't care, output constraint:null.
- Numeric slots (Year, city mpg, highway MPG, MSRP):
  - Use >= for minimum requirements, <= for maximum budget, == only if explicitly exact.
  - value MUST be an integer.
- Categorical slots (Engine Fuel Type, Vehicle Size, Model):
  - op must be == and value string.
Output ONLY JSON.
""".strip()

    obj = gemma_gen_json(AGENT_EXTRACT_ONLY_SYS, user_prompt, max_new_tokens=MAX_NEW_TOKENS_PARSE, retries=3)
    c = obj.get("constraint", None)

    if c is not None:
        c["column"] = slot
        sc = sanitize_constraint(c)
        if sc is not None:
            return sc

    # fallback recovery
    if slot in CATEGORICAL_COLS:
        m = _best_vocab_match(slot, user_answer)
        if m is not None:
            m = canonicalize_categorical(slot, m)
            if m:
                return {"column": slot, "op": "==", "value": m}
        return None

    if slot in NUMERIC_COLS:
        h = _heuristic_numeric(slot, user_answer)
        return sanitize_constraint(h) if h is not None else None

    return None

def parse_base_from_sentence(df: pd.DataFrame, base_sentence: str) -> Dict[str,str]:
    s = base_sentence.lower()
    def match_vocab(col: str) -> str:
        vocab = sorted(df[col].dropna().astype(str).unique().tolist(), key=len, reverse=True)
        for v in vocab:
            if v.lower() in s:
                return v
        raise ValueError(f"Could not parse base field {col} from: {base_sentence}")
    return {
        "Make": match_vocab("Make"),
        "Vehicle Style": match_vocab("Vehicle Style"),
        "Transmission Type": match_vocab("Transmission Type"),
        "Driven_Wheels": match_vocab("Driven_Wheels"),
    }

# ----------------------------
# Run simulation
# ----------------------------
outputs = []

for i, b in enumerate(bench, 1):
    base_query = b["base_query_sentence"]
    persona = b.get("persona", "")

    slots = sorted({c["column"] for c in b.get("additional_constraints", [])})

    history = [{"role":"user", "content": base_query}]
    collected_constraints: List[Dict[str,Any]] = []

    for slot in slots:
        tries = 0
        c = None
        while tries <= MAX_REASKS_PER_SLOT and c is None:
            q = agent_question(slot, base_query, history)
            history.append({"role":"assistant","content": q})

            a = roleplayer_answer(persona, base_query, q)
            history.append({"role":"user","content": a})

            c = extract_constraint(slot, a)
            tries += 1

            if c is None and NO_PREF_PAT.search(a or ""):
                break

        if c is not None:
            collected_constraints.append(c)

    collected_constraints = dedup_constraints(collected_constraints)

    base = parse_base_from_sentence(df, base_query)

    U = df[(df["Make"] == base["Make"]) &
           (df["Transmission Type"] == base["Transmission Type"]) &
           (df["Driven_Wheels"] == base["Driven_Wheels"]) &
           (df["Vehicle Style"] == base["Vehicle Style"])].copy().reset_index(drop=True)

    # Option A: weights after dialogue
    wmap = global_weighting_pass(history, collected_constraints)

    record = {
        "id": i,
        "base_query_sentence": base_query,
        "slots_provided_to_agent": slots,
        "dialogue": history,
        "parsed_base": base,
        "parsed_additional_constraints": collected_constraints,
        "constraint_weights": [
            {"column": p["column"], "op": p["op"], "value": p["value"],
             "weight": float(wmap.get((p["column"], p["op"], str(p["value"])), 0.0))}
            for p in collected_constraints
        ],
        "status": None,
        "relaxed_constraint": None,
        "recommended_car": None
    }

    # Try full query
    full = apply_constraints(df, base, collected_constraints).reset_index(drop=True)
    if len(full) > 0:
        if len(full) == 1:
            record["status"] = "SAT_no_relaxation"
            record["recommended_car"] = row_to_car(full.iloc[0])
        else:
            scores = score_candidates_soft(U, full, collected_constraints, wmap)
            best = pick_best_row(full, scores)
            record["status"] = "SAT_no_relaxation"
            record["recommended_car"] = row_to_car(best.iloc[0])
        outputs.append(record)
        print(f"[{i}] SAT full. Results={len(full)} (constraints={len(collected_constraints)}/{len(slots)})")
        continue

    # UNSAT: relax lowest-weight, then next-lowest, etc.
    if len(collected_constraints) == 0:
        record["status"] = "UNSAT_no_constraints_collected"
        outputs.append(record)
        print(f"[{i}] UNSAT no constraints collected. (slots={len(slots)})")
        continue

    keys = [(p["column"], p["op"], str(p["value"])) for p in collected_constraints]
    keys_sorted = sorted(keys, key=lambda k: float(wmap.get(k, 0.0)))

    found = False
    for relaxed_key in keys_sorted:
        keep = [p for p in collected_constraints if (p["column"], p["op"], str(p["value"])) != relaxed_key]
        cand = apply_constraints(df, base, keep).reset_index(drop=True)
        if len(cand) == 0:
            continue

        for p in collected_constraints:
            if (p["column"], p["op"], str(p["value"])) == relaxed_key:
                record["relaxed_constraint"] = {"column": p["column"], "op": p["op"], "value": p["value"]}
                break

        if len(cand) == 1:
            record["status"] = "SAT_after_relaxation"
            record["recommended_car"] = row_to_car(cand.iloc[0])
            found = True
            break

        # multiple -> renormalize remaining weights
        keep_keys = [(p["column"], p["op"], str(p["value"])) for p in keep]
        s = sum(float(wmap.get(k, 0.0)) for k in keep_keys)
        if s <= 0:
            w_keep = {k: 1.0/len(keep_keys) for k in keep_keys}
        else:
            w_keep = {k: float(wmap.get(k, 0.0))/s for k in keep_keys}

        scores = score_candidates_soft(U, cand, keep, w_keep)
        best = pick_best_row(cand, scores)
        record["status"] = "SAT_after_relaxation"
        record["recommended_car"] = row_to_car(best.iloc[0])
        found = True
        break

    if not found:
        record["status"] = "UNSAT_even_after_relaxation"

    outputs.append(record)
    print(f"[{i}] UNSAT -> {record['status']} (relaxed={record['relaxed_constraint']}) (constraints={len(collected_constraints)}/{len(slots)})")

# Save
with open(AGENT_OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(outputs, f, indent=2, ensure_ascii=False)

print(f"\nWrote outputs to: {AGENT_OUT_PATH}")

# ----------------------------
# Benchmark comparison
# ----------------------------
def ckey(c):
    if c is None:
        return None
    return (c.get("column"), c.get("op"), str(c.get("value")))

def car_key(car):
    if not isinstance(car, dict) or not car:
        return None
    return (car.get("Make"), car.get("Model"), str(car.get("Year")), str(car.get("MSRP")))

bench_by_base = {b["base_query_sentence"]: b for b in bench}

relax_matches = 0
car_matches = 0
n = 0

for a in outputs:
    baseq = a["base_query_sentence"]
    if baseq not in bench_by_base:
        continue
    b = bench_by_base[baseq]
    n += 1

    oracle_relaxed = (b.get("unique_repair_constraint") or b.get("chosen_relaxation") or b.get("relaxed_constraint"))
    agent_relaxed = a.get("relaxed_constraint")

    relax_ok = (ckey(oracle_relaxed) == ckey(agent_relaxed))
    relax_matches += int(relax_ok)

    oracle_car = b.get("recommended_car")
    agent_car = a.get("recommended_car")

    car_ok = relax_ok and (car_key(oracle_car) == car_key(agent_car))
    car_matches += int(car_ok)

    print(f"\n=== {n} ===")
    print("Base:", baseq)
    print("Oracle relaxed:", oracle_relaxed)
    print("Agent  relaxed:", agent_relaxed)
    print("Relax match:", relax_ok)
    print("Oracle car key:", car_key(oracle_car))
    print("Agent  car key:", car_key(agent_car))
    print("Car match:", car_ok)

print("\n=== SUMMARY ===")
print("Compared:", n)
print("Relax matches:", relax_matches, f"({relax_matches/n:.2%})" if n else "")
print("Car matches:", car_matches, f"({car_matches/n:.2%})" if n else "")


In [ ]:
# ============================================================
# Two-LLM simulation (Roleplayer + Agent) + PRP (Pairwise Ranking Prompting) weighting
# ============================================================

import os, json, re
from typing import Any, Dict, List, Optional, Tuple
from itertools import combinations

import numpy as np
import pandas as pd

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from setproctitle import setproctitle
setproctitle("python")

os.environ["CUDA_VISIBLE_DEVICES"] = "4"   # only use physical GPUs 2 and 3
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ----------------------------
# Paths / config
# ----------------------------
BENCH_PATH_CANDS = ["./44fix.json", "/mnt/data/queries_10.json"]
DATA_PATH_CANDS  = ["./data.csv", "/mnt/data/data.csv"]

AGENT_OUT_PATH = "./gamma/Gamma_prp44.json"

MODEL_NAME = "google/gemma-2-9b-it"
SEED = 11
rng = np.random.default_rng(SEED)

EXCLUDE = {"Engine HP", "Engine Cylinders", "Market Category", "Popularity", "Number of Doors"}
BASE_COLS = ["Make", "Transmission Type", "Driven_Wheels", "Vehicle Style"]

NUMERIC_COLS = {"Year", "city mpg", "highway MPG", "MSRP"}
CATEGORICAL_COLS = {"Engine Fuel Type", "Vehicle Size", "Model"}
ALLOWED_EXTRA_COLS = list(NUMERIC_COLS | CATEGORICAL_COLS)
ALLOWED_OPS = {"==", ">=", "<="}

MAX_NEW_TOKENS_Q = 120
MAX_NEW_TOKENS_A = 200
MAX_NEW_TOKENS_PARSE = 240

PRP_EPS = 1e-6  # for safe normalization

# ----------------------------
# Load benchmark + data
# ----------------------------
bench_path = next((p for p in BENCH_PATH_CANDS if os.path.exists(p)), None)
if bench_path is None:
    raise FileNotFoundError("Could not find benchmark json in ./ or /mnt/data/")

data_path = next((p for p in DATA_PATH_CANDS if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError("Could not find data.csv in ./ or /mnt/data/")

with open(bench_path, "r", encoding="utf-8") as f:
    bench = json.load(f)

df0 = pd.read_csv(data_path)
df = df0[[c for c in df0.columns if c not in EXCLUDE]].copy()
if "Engine Fuel Type" in df.columns:
    df["Engine Fuel Type"] = df["Engine Fuel Type"].fillna("Unknown")

# ----------------------------
# Canonicalization helpers
# ----------------------------
def _build_vocab_maps(df: pd.DataFrame, cols: List[str]):
    vocab_list = {}
    vocab_lower_to_orig = {}
    for c in cols:
        if c not in df.columns:
            continue
        vals = df[c].dropna().astype(str).unique().tolist()
        vocab_list[c] = vals
        m = {}
        for v in vals:
            vl = v.strip().lower()
            if vl and vl not in m:
                m[vl] = v
        vocab_lower_to_orig[c] = m
    return vocab_list, vocab_lower_to_orig

VOCAB_LIST, VOCAB_LOWER_TO_ORIG = _build_vocab_maps(df, list(CATEGORICAL_COLS))
MAKE_SET_LOWER = set(df["Make"].dropna().astype(str).str.strip().str.lower().unique().tolist()) if "Make" in df.columns else set()

def canonicalize_categorical(col: str, raw: str) -> str:
    s = (raw or "").strip()
    if s == "" or col not in VOCAB_LOWER_TO_ORIG:
        return s

    low = s.lower()
    m = VOCAB_LOWER_TO_ORIG[col]
    if low in m:
        return m[low]

    s2 = s
    low2 = low
    if col == "Model":
        toks = re.split(r"\s+", s)
        if toks:
            t0 = toks[0].strip().lower()
            if t0 in MAKE_SET_LOWER and len(toks) >= 2:
                s2 = " ".join(toks[1:]).strip()
                low2 = s2.lower()
                if low2 in m:
                    return m[low2]

    def _clean(x: str) -> str:
        x = x.lower()
        x = re.sub(r"[^a-z0-9\s\-\/]", " ", x)
        x = re.sub(r"\b(the|a|an|model|trim|edition)\b", " ", x)
        x = re.sub(r"\s+", " ", x).strip()
        return x

    c_low = _clean(low2)

    best = None
    best_len = -1
    for v in VOCAB_LIST.get(col, []):
        vl = _clean(str(v))
        if not vl:
            continue
        if vl in c_low or c_low in vl:
            if len(vl) > best_len:
                best = v
                best_len = len(vl)
    if best is not None:
        return str(best)

    return s

# ----------------------------
# Gemma 2 load + generation helpers
# ----------------------------
def load_gemma2(model_name: str = MODEL_NAME):
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    dtype_try = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    try:
        mdl = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=dtype_try,
        )
    except Exception:
        mdl = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        )
    mdl.eval()
    return tok, mdl

tok, mdl = load_gemma2(MODEL_NAME)

def _extract_json_robust(text: str) -> dict:
    t = text.strip()
    if "```" in t:
        parts = t.split("```")
        for i in range(len(parts)-1):
            fence = parts[i].strip().lower()
            payload = parts[i+1].strip()
            if "json" in fence or fence == "":
                lines = payload.splitlines()
                if lines and lines[0].strip().lower() == "json":
                    payload = "\n".join(lines[1:]).strip()
                try:
                    return json.loads(payload)
                except Exception:
                    t = payload
                    break
    start = t.find("{")
    if start == -1:
        raise ValueError("No '{' found in model output.")
    depth = 0
    for i in range(start, len(t)):
        ch = t[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return json.loads(t[start:i+1])
    end = t.rfind("}")
    if end != -1 and end > start:
        return json.loads(t[start:end+1])
    raise ValueError("Failed to parse JSON object from model output.")

def _gemma_apply_chat(user_content: str) -> str:
    chat = [{"role": "user", "content": user_content}]
    try:
        return tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    except Exception:
        bos = tok.bos_token or "<bos>"
        return f"{bos}<start_of_turn>user\n{user_content}<end_of_turn>\n<start_of_turn>model\n"

@torch.inference_mode()
def gemma_gen_text(system_prompt: str, user_prompt: str, max_new_tokens: int = 200) -> str:
    combined = (system_prompt or "").strip()
    if combined:
        combined = combined + "\n\n" + (user_prompt or "").strip()
    else:
        combined = (user_prompt or "").strip()

    prompt_text = _gemma_apply_chat(combined)
    inputs = tok(prompt_text, return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(mdl.device) for k, v in inputs.items()}
    if "attention_mask" not in inputs or inputs["attention_mask"] is None:
        inputs["attention_mask"] = (inputs["input_ids"] != tok.pad_token_id).long()

    prompt_len = inputs["input_ids"].shape[-1]
    out = mdl.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
    )
    gen_ids = out[0][prompt_len:]
    return tok.decode(gen_ids, skip_special_tokens=True).strip()

@torch.inference_mode()
def gemma_gen_json(system_prompt: str, user_prompt: str, max_new_tokens: int = 260, retries: int = 3) -> dict:
    strict_suffixes = [
        "",
        "\n\nIMPORTANT: Output MUST start with '{' and end with '}' and contain NOTHING else.",
        "\n\nFINAL WARNING: ONLY output JSON. No markdown. No code fences. No commentary."
    ]
    last_text, last_err = None, None
    for attempt in range(retries):
        sys = (system_prompt or "") + strict_suffixes[min(attempt, len(strict_suffixes)-1)]
        txt = gemma_gen_text(sys, user_prompt, max_new_tokens=max_new_tokens)
        last_text = txt
        try:
            return _extract_json_robust(txt)
        except Exception as e:
            last_err = e
            continue
    raise ValueError(f"Could not parse JSON after {retries} tries. Last error: {last_err}\n\nLast model output:\n{last_text}")

# ----------------------------
# Sanitizers
# ----------------------------
def coerce_int(v) -> Optional[int]:
    if v is None:
        return None
    if isinstance(v, (int, np.integer)):
        return int(v)
    if isinstance(v, float) and np.isfinite(v):
        return int(round(v))
    s = str(v).strip()
    m = re.search(r"-?\d+", s)
    if not m:
        return None
    try:
        return int(m.group(0))
    except Exception:
        return None

def sanitize_constraint(p: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    if not isinstance(p, dict):
        return None
    col, op, val = p.get("column"), p.get("op"), p.get("value")
    if col not in ALLOWED_EXTRA_COLS or op not in ALLOWED_OPS:
        return None
    if col in NUMERIC_COLS:
        if op in (">=", "<=", "=="):
            iv = coerce_int(val)
            if iv is None:
                return None
            return {"column": col, "op": op, "value": iv}
        return None
    if col in CATEGORICAL_COLS:
        if op != "==":
            return None
        sval = str(val).strip()
        if sval == "":
            return None
        sval = canonicalize_categorical(col, sval)
        if sval == "":
            return None
        return {"column": col, "op": "==", "value": sval}
    return None

def dedup_constraints(extras: List[Dict[str,Any]]) -> List[Dict[str,Any]]:
    seen = set()
    out = []
    for p in extras:
        k = (p["column"], p["op"], str(p["value"]))
        if k in seen:
            continue
        seen.add(k)
        out.append(p)
    return out

# ----------------------------
# "Solver" filtering
# ----------------------------
def apply_constraints(df: pd.DataFrame, base: Dict[str,str], extras: List[Dict[str,Any]]) -> pd.DataFrame:
    m = np.ones(len(df), dtype=bool)
    for c in BASE_COLS:
        m &= (df[c].values == base[c])
    for p in extras:
        col, op, val = p["column"], p["op"], p["value"]
        if col not in df.columns:
            return df.iloc[0:0].copy()
        if op == "==":
            m &= (df[col].astype(str).values == str(val))
        elif op == ">=":
            m &= (df[col].values >= int(val))
        elif op == "<=":
            m &= (df[col].values <= int(val))
        else:
            return df.iloc[0:0].copy()
    return df[m].copy()

# ----------------------------
# Scoring/recommendation utilities
# ----------------------------
def normalize_series(x: np.ndarray) -> np.ndarray:
    xmin = float(np.min(x))
    xmax = float(np.max(x))
    if xmax - xmin < 1e-9:
        return np.zeros_like(x, dtype=float)
    return (x - xmin) / (xmax - xmin)

def score_candidates_soft(
    U: pd.DataFrame,
    cand: pd.DataFrame,
    constraints: List[Dict[str,Any]],
    weights: Dict[Tuple[str,str,str], float],
) -> np.ndarray:
    if len(cand) == 0:
        return np.array([], dtype=float)

    scores = np.zeros(len(cand), dtype=float)

    norms = {}
    for col in ["Year","city mpg","highway MPG","MSRP"]:
        if col in U.columns:
            norms[col] = normalize_series(U[col].values)

    cand_pos = cand.index.values

    for p in constraints:
        k = (p["column"], p["op"], str(p["value"]))
        w = float(weights.get(k, 0.0))
        col, op, val = p["column"], p["op"], p["value"]

        if op == "==":
            sat = (cand[col].astype(str).values == str(val)).astype(float)
            scores += w * sat
        elif op == ">=":
            if col in norms:
                scores += w * norms[col][cand_pos]
            else:
                scores += w * (cand[col].values >= int(val)).astype(float)
        elif op == "<=":
            if col in norms:
                scores += w * (1.0 - norms[col][cand_pos])
            else:
                scores += w * (cand[col].values <= int(val)).astype(float)

    return scores

def recommend_from_candidates(cand: pd.DataFrame) -> Optional[Dict[str,Any]]:
    if len(cand) == 0:
        return None
    row = cand.iloc[0]
    def gi(k):
        return int(row[k]) if k in cand.columns and pd.notna(row[k]) else None
    return {
        "Make": row.get("Make", None),
        "Model": row.get("Model", None),
        "Year": gi("Year"),
        "Engine Fuel Type": row.get("Engine Fuel Type", None),
        "Transmission Type": row.get("Transmission Type", None),
        "Driven_Wheels": row.get("Driven_Wheels", None),
        "Vehicle Size": row.get("Vehicle Size", None),
        "Vehicle Style": row.get("Vehicle Style", None),
        "city mpg": gi("city mpg"),
        "highway MPG": gi("highway MPG"),
        "MSRP": gi("MSRP"),
    }

# ----------------------------
# PRP: Pairwise Ranking Prompting (logprob comparator)
# ----------------------------
PRP_SYS = (
    "You are ranking user requirements by importance. "
    "Choose which requirement is MORE IMPORTANT to the user to keep satisfied. "
    "Answer only with 'A' or 'B'."
)

def format_constraint_nl(p: Dict[str,Any]) -> str:
    col, op, val = p["column"], p["op"], p["value"]
    if op == "==":
        return f"{col} is '{val}'"
    if col == "MSRP" and op == "<=":
        return f"MSRP is at most ${int(val):,}"
    if col in ("city mpg", "highway MPG") and op == ">=":
        return f"{col} is at least {int(val)}"
    if col == "Year" and op == ">=":
        return f"Year is {int(val)} or newer"
    return f"{col} {op} {val}"

@torch.inference_mode()
def _continuation_logprob(prompt_text: str, continuation: str) -> float:
    # IMPORTANT: In this function, prompt_text is already the full formatted chat prompt
    prompt_ids = tok(prompt_text, return_tensors="pt", add_special_tokens=False).input_ids.to(mdl.device)
    cont_ids = tok(continuation, return_tensors="pt", add_special_tokens=False).input_ids.to(mdl.device)

    full_ids = torch.cat([prompt_ids, cont_ids], dim=1)
    attn = torch.ones_like(full_ids)

    out = mdl(input_ids=full_ids, attention_mask=attn)
    logits = out.logits

    prompt_len = prompt_ids.shape[1]
    cont_len = cont_ids.shape[1]

    logp = 0.0
    for i in range(cont_len):
        pos = prompt_len + i
        token_id = int(full_ids[0, pos].item())
        prev_logits = logits[0, pos-1]
        lp = torch.log_softmax(prev_logits, dim=-1)[token_id].item()
        logp += lp
    return float(logp)

def prp_compare_once(history_text: str, base_query: str, A: Dict[str,Any], B: Dict[str,Any]) -> str:
    # Inline "system" as part of user message for Gemma
    user_prompt = f"""
Instructions:
{PRP_SYS}

Base query:
{base_query}

Conversation (user + assistant):
{history_text}

Which requirement is MORE IMPORTANT to the user?

A: {format_constraint_nl(A)}
B: {format_constraint_nl(B)}

Answer (A/B):
""".strip()

    prompt_text = _gemma_apply_chat(user_prompt)

    # Compare probabilities of single-token-ish continuations.
    # Keep leading space for better tokenization stability.
    lpA = _continuation_logprob(prompt_text, " A")
    lpB = _continuation_logprob(prompt_text, " B")
    return "A" if lpA >= lpB else "B"

def prp_pairwise_winner(history_text: str, base_query: str, A: Dict[str,Any], B: Dict[str,Any]) -> str:
    r1 = prp_compare_once(history_text, base_query, A, B)
    r2 = prp_compare_once(history_text, base_query, B, A)
    r2_in_original = "A" if r2 == "B" else "B"
    if r1 == r2_in_original:
        return r1
    return "TIE"

def prp_weights(history: List[Dict[str,str]], base_query: str, constraints: List[Dict[str,Any]]) -> Dict[Tuple[str,str,str], float]:
    if len(constraints) == 0:
        return {}
    if len(constraints) == 1:
        p = constraints[0]
        return {(p["column"], p["op"], str(p["value"])): 1.0}

    history_text = "\n".join([f"{m['role'].upper()}: {m['content']}" for m in history])

    scores = { (p["column"], p["op"], str(p["value"])): 0.0 for p in constraints }
    by_key = { (p["column"], p["op"], str(p["value"])): p for p in constraints }

    keys = list(by_key.keys())
    for i, j in combinations(range(len(keys)), 2):
        ki, kj = keys[i], keys[j]
        pi, pj = by_key[ki], by_key[kj]
        w = prp_pairwise_winner(history_text, base_query, pi, pj)
        if w == "A":
            scores[ki] += 1.0
        elif w == "B":
            scores[kj] += 1.0
        else:
            scores[ki] += 0.5
            scores[kj] += 0.5

    total = sum(v + PRP_EPS for v in scores.values())
    weights = {k: float((v + PRP_EPS) / total) for k, v in scores.items()}
    return weights

# ----------------------------
# Two-LLM simulation: Roleplayer answers, Agent asks + extracts constraints
# ----------------------------
ROLEPLAYER_SYS = (
    "You are ROLEPLAYER_USER. You ONLY know the provided persona and the base query. "
    "Answer the assistant's question using ONLY information that is stated or clearly implied in the persona. "
    "IMPORTANT: When the assistant is clarifying a constraint, you should be helpful and specific: "
    "if the persona provides ANY relevant value (e.g., a specific model name, a year minimum, a budget cap), "
    "you MUST state that value explicitly in your answer. "
    "Do NOT say 'no preference' or omit the value if the persona mentions it. "
    "Only say you have no preference if the persona truly contains no information about that attribute. "
    "If the assistant's question is a bit vague, still provide the most relevant constraint value(s) from the persona. "
    "Do not invent details that are not in the persona."
)

AGENT_ASK_SYS = (
    "You are CLARIFYING_AGENT. Ask ONE short question to learn the user's requirement for the given slot/field. "
    "Aim for a concrete value when possible. Do not propose relaxations. Do not ask multiple questions at once."
)

AGENT_EXTRACT_SYS = (
    "You are a strict JSON generator. Extract the constraint value from the user's answer for the given slot. "
    "Output ONLY JSON."
)

def agent_question(slot: str, base_query: str, history: List[Dict[str,str]]) -> str:
    hist_txt = "\n".join([f"{m['role'].upper()}: {m['content']}" for m in history[-6:]])
    user_prompt = f"""
{AGENT_ASK_SYS}

Base query: {base_query}

Conversation so far:
{hist_txt}

Slot to clarify: {slot}

Ask ONE question to determine the user's requirement for {slot}. Keep it concise and concrete.
""".strip()
    return gemma_gen_text("", user_prompt, max_new_tokens=MAX_NEW_TOKENS_Q)

def roleplayer_answer(persona: str, base_query: str, question: str) -> str:
    user_prompt = f"""
{ROLEPLAYER_SYS}

Base query the user started with:
{base_query}

Persona (all you know):
{persona}

Assistant asked:
{question}

Answer as the user, using only persona information.
Reminder: If the persona includes a concrete value relevant to this question, explicitly state it (don't omit it).
""".strip()
    return gemma_gen_text("", user_prompt, max_new_tokens=MAX_NEW_TOKENS_A)

def extract_constraint(slot: str, user_answer: str) -> Optional[Dict[str,Any]]:
    user_prompt = f"""
{AGENT_EXTRACT_SYS}

Slot: {slot}
User answer: {user_answer}

Return JSON:
{{
  "constraint": null OR {{"column": "{slot}", "op": "=="|">="|"<="| "==", "value": string|number}}
}}

Rules:
- If user has no preference / doesn't care, output constraint:null.
- If slot is numeric (Year, city mpg, highway MPG, MSRP):
  - Use >= for minimum requirements, <= for maximum budget, == only if explicitly exact.
  - value MUST be an integer.
- If slot is categorical (Engine Fuel Type, Vehicle Size, Model):
  - Use op == and value as string.
Output ONLY JSON.
""".strip()
    obj = gemma_gen_json("", user_prompt, max_new_tokens=MAX_NEW_TOKENS_PARSE, retries=3)
    c = obj.get("constraint", None)
    if c is None:
        return None
    c["column"] = slot
    return sanitize_constraint(c)

def parse_base_from_sentence(df: pd.DataFrame, base_sentence: str) -> Dict[str,str]:
    s = base_sentence.lower()
    def match_vocab(col: str) -> str:
        vocab = sorted(df[col].dropna().astype(str).unique().tolist(), key=len, reverse=True)
        for v in vocab:
            if v.lower() in s:
                return v
        raise ValueError(f"Could not parse base field {col} from: {base_sentence}")
    return {
        "Make": match_vocab("Make"),
        "Vehicle Style": match_vocab("Vehicle Style"),
        "Transmission Type": match_vocab("Transmission Type"),
        "Driven_Wheels": match_vocab("Driven_Wheels"),
    }

# ----------------------------
# Run simulation for all benchmark queries
# ----------------------------
outputs = []

for i, b in enumerate(bench, 1):
    base_query = b["base_query_sentence"]
    persona = b.get("persona", "")

    slots = sorted({c["column"] for c in b.get("additional_constraints", [])})

    history = [{"role":"user", "content": base_query}]
    collected_constraints: List[Dict[str,Any]] = []

    for slot in slots:
        q = agent_question(slot, base_query, history)
        history.append({"role":"assistant","content": q})

        a = roleplayer_answer(persona, base_query, q)
        history.append({"role":"user","content": a})

        c = extract_constraint(slot, a)
        if c is not None:
            collected_constraints.append(c)

    collected_constraints = dedup_constraints(collected_constraints)

    base = parse_base_from_sentence(df, base_query)

    U = df[(df["Make"] == base["Make"]) &
           (df["Transmission Type"] == base["Transmission Type"]) &
           (df["Driven_Wheels"] == base["Driven_Wheels"]) &
           (df["Vehicle Style"] == base["Vehicle Style"])].copy().reset_index(drop=True)

    wmap = prp_weights(history, base_query, collected_constraints)

    record = {
        "id": i,
        "base_query_sentence": base_query,
        "slots_provided_to_agent": slots,
        "dialogue": history,
        "parsed_base": base,
        "parsed_additional_constraints": collected_constraints,
        "constraint_weights": [
            {"column": p["column"], "op": p["op"], "value": p["value"],
             "weight": float(wmap.get((p["column"], p["op"], str(p["value"])), 0.0))}
            for p in collected_constraints
        ],
        "status": None,
        "relaxed_constraint": None,
        "recommended_car": None
    }

    # 1) Satisfy-first
    full = apply_constraints(df, base, collected_constraints)
    if len(full) > 0:
        cand = full.reset_index(drop=True)
        if len(cand) == 1:
            record["status"] = "SAT_no_relaxation"
            record["recommended_car"] = recommend_from_candidates(cand)
        else:
            scores = score_candidates_soft(U, cand, collected_constraints, wmap)
            msrp = cand["MSRP"].values.astype(float) if "MSRP" in cand.columns else np.zeros(len(cand))
            best_idx = np.lexsort((msrp, -scores))[0]
            best = cand.iloc[[best_idx]].reset_index(drop=True)
            record["status"] = "SAT_no_relaxation"
            record["recommended_car"] = recommend_from_candidates(best)

        outputs.append(record)
        print(f"[{i}] SAT with full query. Results={len(full)}")
        continue

    # 2) UNSAT: relax constraints in increasing-weight order until results
    if len(collected_constraints) == 0:
        record["status"] = "UNSAT_no_constraints_collected"
        outputs.append(record)
        print(f"[{i}] UNSAT and no constraints collected.")
        continue

    keys = [(p["column"], p["op"], str(p["value"])) for p in collected_constraints]
    keys_sorted = sorted(keys, key=lambda k: float(wmap.get(k, 0.0)))

    found = False
    for relaxed_key in keys_sorted:
        keep = [p for p in collected_constraints if (p["column"], p["op"], str(p["value"])) != relaxed_key]
        cand = apply_constraints(df, base, keep).reset_index(drop=True)

        if len(cand) == 0:
            continue

        for p in collected_constraints:
            if (p["column"], p["op"], str(p["value"])) == relaxed_key:
                record["relaxed_constraint"] = {"column": p["column"], "op": p["op"], "value": p["value"]}
                break

        if len(cand) == 1:
            record["status"] = "SAT_after_relaxation"
            record["recommended_car"] = recommend_from_candidates(cand)
            found = True
            break

        keep_keys = [(p["column"], p["op"], str(p["value"])) for p in keep]
        s = sum(float(wmap.get(k, 0.0)) for k in keep_keys)
        if s <= 0:
            w_keep = {k: 1.0/len(keep_keys) for k in keep_keys}
        else:
            w_keep = {k: float(wmap.get(k, 0.0))/s for k in keep_keys}

        scores = score_candidates_soft(U, cand, keep, w_keep)
        msrp = cand["MSRP"].values.astype(float) if "MSRP" in cand.columns else np.zeros(len(cand))
        best_idx = np.lexsort((msrp, -scores))[0]
        best = cand.iloc[[best_idx]].reset_index(drop=True)

        record["status"] = "SAT_after_relaxation"
        record["recommended_car"] = recommend_from_candidates(best)
        found = True
        break

    if not found:
        record["status"] = "UNSAT_even_after_relaxation"
        record["recommended_car"] = None

    outputs.append(record)
    print(f"[{i}] UNSAT -> {record['status']} (relaxed={record['relaxed_constraint']})")

with open(AGENT_OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(outputs, f, indent=2, ensure_ascii=False)

print(f"\nWrote outputs to: {AGENT_OUT_PATH}")

# ============================================================
# Simple benchmark comparison (relaxed constraint + recommended car)
# ============================================================
def ckey(c):
    if c is None:
        return None
    return (c.get("column"), c.get("op"), str(c.get("value")))

def car_key(car):
    if not isinstance(car, dict) or not car:
        return None
    return (car.get("Make"), car.get("Model"), str(car.get("Year")), str(car.get("MSRP")))

bench_by_base = {b["base_query_sentence"]: b for b in bench}

relax_matches = 0
car_matches = 0
n = 0

for a in outputs:
    baseq = a["base_query_sentence"]
    if baseq not in bench_by_base:
        print(f"\n[SKIP] Not found in bench: {baseq}")
        continue

    b = bench_by_base[baseq]
    n += 1

    oracle_relaxed = (
        b.get("unique_repair_constraint")
        or b.get("chosen_relaxation")
        or b.get("relaxed_constraint")
    )
    agent_relaxed = a.get("relaxed_constraint")

    relax_ok = (ckey(oracle_relaxed) == ckey(agent_relaxed))
    relax_matches += int(relax_ok)

    oracle_car = b.get("recommended_car")
    agent_car = a.get("recommended_car")

    # penalty: car match cannot count unless relaxation matches
    car_ok = relax_ok and (car_key(oracle_car) == car_key(agent_car))
    car_matches += int(car_ok)

    print(f"\n=== {n} ===")
    print("Base:", baseq)
    print("Oracle relaxed:", oracle_relaxed)
    print("Agent  relaxed:", agent_relaxed)
    print("Relax match:", relax_ok)
    print("Oracle car key:", car_key(oracle_car))
    print("Agent  car key:", car_key(agent_car))
    print("Car match:", car_ok)

print("\n=== SUMMARY ===")
print("Compared:", n)
print("Relax matches:", relax_matches, f"({relax_matches/n:.2%})" if n else "")
print("Car matches:", car_matches, f"({car_matches/n:.2%})" if n else "")
